# 量子コンピューティング I ── 第2回 演習ノートブック
## 量子ビットと量子もつれ

本ノートブックでは、第2回講義の内容に対応した演習を行う。

- セクション2-3: 量子ビットと単一量子ゲート（演習1に対応）
- セクション4: CXゲートとベル状態（演習2に対応）
- セクション5: 位相の観察（演習3に対応）

推奨環境: Google Colab / ローカルのJupyter環境（Python 3.10以上）

In [ ]:
# セットアップ
# まずはQiskitをこのjupyter notebook環境にインストールしましょう
# Google Colabでは毎回このセルを実行すること。ローカル環境では初回のみ。
!pip install -q qiskit[visualization] qiskit-aer

In [ ]:
import qiskit
print(f"Qiskit version: {qiskit.__version__}")

In [ ]:
# 共通インポートとユーティリティ関数
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer import AerSimulator # qiskit_aerという量子回路をシミュレートするためのライブラリ
from qiskit import transpile #量子回路のシミュレーション前にトランスパイル（コンパイル）する必要があります。

# 簡単のために回路のシミュレーションを関数にしておきます。自分の書いている量子回路の振る舞いを知りたい時は適宜実行してください。
def run_qc_on_aer(qc, shots=1024):
    """量子回路をシミュレータで実行し、測定結果を返す。"""
    simulator = AerSimulator()
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=shots).result()
    return result.get_counts()

---
# 練習 1: 量子ビットの準備と測定

量子レジスタと古典レジスタを準備し、空の量子回路を作成する。
量子ビットは生成時に $|0\rangle$ に初期化される。

In [ ]:
# 1量子ビット、1古典ビットの回路を作成してみましょう
qr = QuantumRegister(1) # 1量子ビットの量子レジスタを用意します。このとき、すべての量子ビットは|0>に初期化されています。
cr = ClassicalRegister(1) # 1ビットの古典レジスタを用意します。
qc = QuantumCircuit(qr, cr) # qrとcrからなるシステムを用意します。
qc.draw(output='mpl') # 作った量子回路を描画してみましょう。

In [ ]:
# 状態ベクトルを取得してみましょう。
state = Statevector(qc)
print('状態ベクトル:', state)
# さらにブロッホ球で表示してみましょう。初期状態は |0> なので、北極に点があるはずです
plot_bloch_multivector(state)

In [ ]:
# 測定を追加して量子ビットの中身を読み出してみましょう。
qc.measure(qr[0], cr[0])
qc.draw(output='mpl')

In [ ]:
# シミュレータで1024回実行してみましょう。古典出力をサンプリングすることができます。
# |0> 状態を測定しているので、0が1024回出るはずです
counts = run_qc_on_aer(qc)
print(counts)

---
# 練習 2: 単一量子ビットゲート

主要な単一量子ビットゲートを一つずつ確認する。
各ゲートを適用した後の状態をブロッホ球で観察し、測定結果を確認すること。

## Xゲート（ビット反転）

Xゲートはブロッホ球のX軸周りの $\pi$ 回転であり、古典のNOTゲートに相当する。

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$$

$X|0\rangle = |1\rangle$, $X|1\rangle = |0\rangle$

In [ ]:
# 先ほどは{'0': 1024} のような結果が得られたはずです。次に、|1>状態を作って古典出力を確かめてみましょう。

# |0> に X を適用して |1> を作る
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)
qc.x(qr[0])
qc.draw(output='mpl')

In [ ]:
# ブロッホ球で確認する。南極（|1>）に移動しているはず
state = Statevector(qc)
plot_bloch_multivector(state)

In [ ]:
# 無事|1>状態を準備できたら、先ほどと同様に測定してみましょう。1が100%になるでしょうか？
qc.measure(qr[0], cr[0])
#qc.draw(output='mpl')
counts = run_qc_on_aer(qc)
print(counts)

---
# 演習 1: |-> を作成し、Z測定で比較しよう

## 準備: Hゲート（アダマールゲート）で|+>を作成する

Hゲートはブロッホ球のX軸とZ軸の中間の軸周りの $\pi$ 回転であり、
重ね合わせ状態を生成する最も重要なゲートである。

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

$H|0\rangle = |+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$

$H|1\rangle = |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$

$|+\rangle$ と $|-\rangle$ は測定確率が同じ（50/50）だが、位相が異なる。
この違いがどういう意味を持つかは、この後の演習で確認する。

In [ ]:
# |0> に H を適用して |+> を作る
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)
qc.h(qr[0])
qc.draw(output='mpl')

In [ ]:
# ブロッホ球で確認する。赤道上にいるはず
state = Statevector(qc)
plot_bloch_multivector(state)

In [ ]:
# 測定する。0と1がほぼ半々になる
qc.measure(qr[0], cr[0])
counts = run_qc_on_aer(qc)
plot_histogram(counts)

In [ ]:
counts_plus_Z = run_qc_on_aer(qc)
print("|+> Z測定:", counts_plus_Z)

## 本題: |-> を作成し、Z測定で|+>と比較しよう

In [ ]:
# |-> を 作成する
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)

# 1. X|0> = |1> を準備する
# ここでゲート追加

# 2. H|1> = |-> を準備する

# H|1> = |-> で作成できる
# ここでゲート追加

# ブロッホ球で確認する。赤道上の、|+>とは反対側にいるはず
state = Statevector(qc)
plot_bloch_multivector(state)


In [ ]:
# Z測定し、出力を確認する
qc.measure(qr[0], cr[0])
counts_minus_Z = run_qc_on_aer(qc)
print("|-> Z測定:", counts_minus_Z)

# 両方とも 50/50 になる。Z測定では |+> と |-> を区別できない

## 練習 3: Zゲート（位相反転）

Zゲートはブロッホ球のZ軸周りの $\pi$ 回転である。

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

$Z|0\rangle = |0\rangle$, $Z|1\rangle = -|1\rangle$

$|0\rangle$ に対しては何も変わらない。$|1\rangle$ の符号（位相）だけが反転する。

In [ ]:
# |0> に Z を適用する
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)
qc.z(qr[0])
qc.measure(qr[0], cr[0])
counts = run_qc_on_aer(qc)
print(counts)
# 0が100%。Z|0> = |0> なので測定結果は変わらない

---
## 演習 3: |+>と|->を見分ける測定 (X測定)

**準備： 演習 1などを参考に、|+>か|->を用意しよう**
- H |0>  → |+>
- H X |0> → H |1> → |->

**本題： 準備した状態に、Hゲートを掛けてから測定してみよう**
- X測定は、測定の直前にHゲートを挟むことで実現する。
$H$ が $\{|+\rangle, |-\rangle\}$ 基底を $\{|0\rangle, |1\rangle\}$ 基底に変換するためである。


以下のセルを実行し、結果を予想してから確認すること。

※ここで、Zゲートを|+>と|->の切り替えに用いている。（なぜ上手くいくのか考えてみよう。）

In [ ]:
# 演習3: 位相情報を見分ける
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)

qc.h(qr[0])   # Step 1: |0> -> |+>
#qc.z(qr[0])   # Step 2: |+> -> |->  (位相だけ変わる)

qc.h(qr[0])   # Step 3: |?> -> ??? （Z軸とX軸を入れ替える）
qc.measure(qr[0], cr[0]) # X測定

qc.draw(output='mpl')

In [ ]:
# 実行して結果を確認する
counts = run_qc_on_aer(qc)
print(counts)
plot_histogram(counts)

### 結果のまとめ

| 状態 | Z測定 | X測定 |
|------|-------|-------|
| $|+\rangle$ | 50/50 | 0 が 100% |
| $|-\rangle$ | 50/50 | 1 が 100% |

Z測定では区別できなかった $|+\rangle$ と $|-\rangle$ が、
X測定（Hを挟む）では完全に区別できた。

位相は「見えない情報」ではない。測定基底を変えれば確実に観測できる。

この原理は以下で再び登場する:
- 第3-4回: Groverのアルゴリズムにおけるオラクル（位相反転）とDiffuser（基底変換）
- 第5回: CHSH不等式の実験における測定基底の選択

### 演習 3の補足： ゲートの等価変換
結果は  
Zゲートを無視すると 0 が 100%  
Zゲートを実行すると 1 が 100% になったはずである。


Zゲートを実行した場合の状態変化を追うと：
1. $H|0\rangle = |+\rangle = (|0\rangle + |1\rangle)/\sqrt{2}$
2. $Z|+\rangle = |{-}\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$ （位相だけが変わった）
3. $H|{-}\rangle = |1\rangle$ （位相の違いが測定結果に現れた）

結論: $HZH = X$ と言える

Zゲート単体では測定結果を変えない（位相操作のみ）。  
しかしHで挟むと、位相の違いが測定結果の違いとして現れる。  
これが量子アルゴリズムでも多用される手法で、
Groverのアルゴリズム（#3-4）でも利用する。

### 発展: S, Tゲート（Z軸回転）

Z軸周りの回転ゲートには、回転角の異なるバリエーションがある。
ブロッホ球の経度（位相）を操作するゲートである。

| ゲート | 行列 | 回転角 |
|--------|------|--------|
| Z | $\text{diag}(1, -1)$ | $\pi$ (180°) |
| S | $\text{diag}(1, i)$ | $\pi/2$ (90°) |
| T | $\text{diag}(1, e^{i\pi/4})$ | $\pi/4$ (45°) |

$S^\dagger$, $T^\dagger$ はそれぞれ逆方向の回転となる。

In [ ]:
# H + S で |Y+> 状態を作り、ブロッホ球で確認する
qr = QuantumRegister(1)
qc = QuantumCircuit(qr)
qc.h(qr[0])
qc.s(qr[0])
state = Statevector(qc)
plot_bloch_multivector(state)
# Y軸の正方向（経度90度）に移動しているはず

（余裕があれば）H + $S^\dagger$ や H + T でもブロッホ球を確認してみよう。  
ベクトルは予想した位置にあろうだろうか？

---
# セクション 4: CXゲートとベル状態

2量子ビットゲートであるCXゲート（CNOT）を使い、量子もつれ状態（ベル状態）を生成する。

## CXゲート（制御NOT / CNOT）

CXゲートは2量子ビットに作用する。
制御ビットが $|1\rangle$ のとき、ターゲットビットを反転する。
制御ビットが $|0\rangle$ のときは何もしない。

$$\text{CX}|00\rangle = |00\rangle, \quad \text{CX}|01\rangle = |01\rangle$$
$$\text{CX}|10\rangle = |11\rangle, \quad \text{CX}|11\rangle = |10\rangle$$

古典のXORに対応する可逆演算である。

In [ ]:
# 練習: CXゲートの動作確認: 制御ビットが |0> の場合
qr = QuantumRegister(2)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

# qr[0]が制御、qr[1]がターゲット
# 制御が |0> なので、ターゲットは変化しない
qc.cx(qr[0], qr[1])
qc.measure(qr, cr)
counts = run_qc_on_aer(qc)
print("制御=|0>:", counts)  # 00が100%

In [ ]:
# 練習: CXゲートの動作確認: 制御ビットが |1> の場合
qr = QuantumRegister(2)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

qc.x(qr[0])  # 制御ビットを |1> にする
qc.cx(qr[0], qr[1])
qc.measure(qr, cr)
counts = run_qc_on_aer(qc)
print("制御=|1>:", counts)  # 11が100%（ターゲットが反転）

## 演習 4: ベル状態の生成

制御ビットを重ね合わせ状態にしてからCXを適用すると、量子もつれが生まれる。

$$|00\rangle \xrightarrow{H \otimes I} \frac{|0\rangle + |1\rangle}{\sqrt{2}} \otimes |0\rangle \xrightarrow{\text{CX}} \frac{|00\rangle + |11\rangle}{\sqrt{2}} = |\Phi^+\rangle$$

測定結果は 00 か 11 のみ。01 や 10 は出ない。

In [ ]:
# ベル状態 |Phi+> を生成する
qr = QuantumRegister(2)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

#ここでqr[0]にHゲートをかける          # 制御ビットを重ね合わせにする

qc.cx(qr[0], qr[1])  # CXで条件反転 → もつれが生まれる
qc.measure(qr, cr)
qc.draw(output='mpl')

In [ ]:
# 実行して結果を確認する
counts = run_qc_on_aer(qc)
print(counts)
plot_histogram(counts)
# うまく実装していれば00と11のみ。01や10は出ない

## |++> 状態の場合

$|{++}\rangle = |+\rangle \otimes |+\rangle$ は各量子ビットが独立な積状態であり、
ベル状態とは全く異なる。
1量子ビットだけ測定した場合はどちらも 50/50 であり区別できないが、
2量子ビットの相関を見ると違いが明らかになる。

In [ ]:
# |++> 状態を作って測定する
qr = QuantumRegister(2)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

qc.h(qr[0])
qc.h(qr[1])  # CXではなくHを両方にかける → 独立な重ね合わせ
qc.measure(qr, cr)
counts = run_qc_on_aer(qc)
print(counts)
plot_histogram(counts)
# 00, 01, 10, 11 がほぼ均等に出る（もつれていない）

### 考察
上の |++> 状態の測定結果と比較し、ベル状態との違いを説明せよ。

---
# 参考: Yゲートとブロッホ球上の回転

以下は必須ではない。
Yゲートとその回転（Ry）を使ったブロッホ球上の状態変化を確認したい場合に参照すること。

## Yゲート

Yゲートはブロッホ球のY軸周りの $\pi$ 回転である。

$$Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}$$

In [ ]:
# Yゲート
qr = QuantumRegister(1)
cr = ClassicalRegister(1)
qc = QuantumCircuit(qr, cr)
qc.y(qr[0])
qc.measure(qr[0], cr[0])
counts = run_qc_on_aer(qc)
print(counts)

RYゲートを使うと任意の角度で回転させることもできる

In [ ]:
# Ry回転でブロッホ球上の状態変化を見る
import numpy as np

qr = QuantumRegister(1)
qc = QuantumCircuit(qr)
qc.ry(np.pi / 3, qr[0])  # pi/3 だけY軸周りに回転
state = Statevector(qc)
plot_bloch_multivector(state)

# 参考: 4種のベル状態

ベル状態は入力を変えるだけで4種類作れる。
以下を実装し、それぞれの測定結果を確認せよ。


| 名前 | 状態 | 入力 |
|------|:------:|------|
| $|\Phi^+\rangle$ | $(|00\rangle + |11\rangle)/\sqrt{2}$ | $|00\rangle$ |
| $|\Phi^-\rangle$ | $(|00\rangle - |11\rangle)/\sqrt{2}$ | $|10\rangle$ |
| $|\Psi^+\rangle$ | $(|01\rangle + |10\rangle)/\sqrt{2}$ | $|01\rangle$ |
| $|\Psi^-\rangle$ | $(|01\rangle - |10\rangle)/\sqrt{2}$ | $|11\rangle$ |


In [ ]:
# 演習2-3: 4種のベル状態を作る
# 以下に回路を構成して実行すること

# 例: |Phi-> を作る場合（入力 |10>）
qr = QuantumRegister(2)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

qc.x(qr[0])          # 入力を |10> にする
qc.h(qr[0])
qc.cx(qr[0], qr[1])
qc.measure(qr, cr)

counts = run_qc_on_aer(qc)
print(counts)
plot_histogram(counts)

### ベル状態を見分ける測定

$|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ と
$|\Phi^-\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$ は、
$|+\rangle$ と $|-\rangle$ の関係と同様に、位相だけが異なる。  
適切な測定基底を選ぶことで区別できる  
4種類全てを見分けるには **ベル基底** と呼ばれる特殊な基底が必要になる。

In [ ]:
# 演習3-5: |Phi+> と |Phi-> を区別する測定を考えて実装せよ
# 両方の量子ビットにHをかけてから測定するとどうなるか？

# --- ここに回路を書く ---